# ZeeChecker: fine-tune the AI rules (LoRA)

1. **Runtime → Change runtime type → T4 GPU**
2. Add a Hugging Face **write** token as a Colab secret named `HF_TOKEN` (key icon on the left). Never paste it into a cell.
3. Run the cells from top to bottom. Upload `train.jsonl`, `val.jsonl` and `test.jsonl` when asked (made-up reports only, **no real reports**).

In [ ]:
%pip install -q "trl==1.18.2" "peft==0.22.0" "transformers==5.24.0"
%pip install -q "llama-cpp-python==0.3.36" --only-binary llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
%pip install -q python-dotenv

If Colab asks to restart the session after the install, restart it and continue with the next cell.

In [ ]:
import torch

assert torch.cuda.is_available(), "Choose a T4 GPU in Runtime -> Change runtime type"
print(torch.cuda.get_device_name())

# The app code to score with. Before its PR is merged, write the PR's branch here
BRANCH = "main"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
REPO_NAME = "zeechecker-qwen2.5-0.5b-GGUF"
GGUF_FILE = "zeechecker-q8_0.gguf"

In [ ]:
from pathlib import Path

from google.colab import files

NAMES = {"train.jsonl", "val.jsonl", "test.jsonl"}
for name in NAMES:
    Path(name).unlink(missing_ok=True)  # or Colab renames the new file to "train (1).jsonl"
uploaded = files.upload()
assert NAMES <= set(uploaded), "Upload all three files together, with exactly these names"

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})
data = data.select_columns(["prompt", "completion"])
print(data)
print(data["train"][0]["prompt"][0]["content"][-600:])
print(data["train"][0]["completion"])

## Baseline: score the untrained model

Uses the app's own check code and the same kind of model file (GGUF) as the app. The seconds per check are Colab's, not the app's. The test examples are never used for training or for choosing the best version.

In [ ]:
import json

import yaml

# Deleted first, because git can't clone into an existing folder and would keep the old code
!rm -rf app && git clone -q --depth 1 --branch {BRANCH} https://github.com/mike1208-dev/team-doc-checker app

assert Path("app/training/evaluate.py").exists(), f"{BRANCH} has no training code. Change BRANCH"
rules = yaml.safe_load(Path("app/rules/daily_report.yaml").read_text(encoding="utf-8"))["rules"]
asked = {json.loads(line)["question"] for line in Path("test.jsonl").read_text(encoding="utf-8").split("\n") if line.strip()}
assert asked <= {rule.get("question") for rule in rules}, f"{BRANCH} asks other questions than test.jsonl. Change BRANCH or make new files"

!cd app && LOCAL_MODEL_REPO=Qwen/Qwen2.5-0.5B-Instruct-GGUF LOCAL_MODEL_FILE=qwen2.5-0.5b-instruct-q8_0.gguf python -m training.evaluate ../test.jsonl --name untrained

## Train

Only the answer counts for the loss, not the prompt. The version with the lowest validation loss is kept.

In [ ]:
from peft import LoraConfig
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

trainer = SFTTrainer(
    model=BASE_MODEL,
    args=SFTConfig(
        output_dir="checkpoints",
        model_init_kwargs={"dtype": "float32"},  # with fp16 below; loading in fp16 breaks training on a T4
        fp16=True,
        bf16=False,
        num_train_epochs=4,
        learning_rate=2e-4,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        max_length=4096,
        completion_only_loss=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=5,
        report_to="none",
        seed=42,
    ),
    train_dataset=data["train"],
    eval_dataset=data["val"],
    processing_class=tokenizer,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
        task_type="CAUSAL_LM",
    ),
)
trainer.train()

## Quick look at a few answers (not a score)

In [ ]:
model = trainer.model.eval()
for row in data["val"].select(range(min(4, len(data["val"])))):
    inputs = tokenizer.apply_chat_template(row["prompt"], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    print("expected:", row["completion"][0]["content"])
    print("model:   ", tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True), "\n")

## Merge and make the model file for the app (GGUF)

In [ ]:
merged = trainer.model.merge_and_unload().to(torch.float16)
merged.save_pretrained("merged")
tokenizer.save_pretrained("merged")

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
%pip install -q gguf sentencepiece
!python llama.cpp/convert_hf_to_gguf.py merged --outfile {GGUF_FILE} --outtype q8_0
!ls -lh {GGUF_FILE}

## Score the trained model, the same way as the baseline

In [ ]:
!cd app && LOCAL_MODEL_PATH=../{GGUF_FILE} python -m training.evaluate ../test.jsonl --name trained

In [ ]:
import pandas as pd

scores = {name: pd.read_csv(f"app/data/results/{name}_test_scores.csv", index_col=0).loc["ALL"] for name in ("untrained", "trained")}
display(pd.DataFrame(scores))

!zip -qr results.zip app/data/results
files.download("results.zip")

## Save to a private Hugging Face repository

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_TOKEN"))
repo_id = f"{api.whoami()['name']}/{REPO_NAME}"
api.create_repo(repo_id, private=True, exist_ok=True)
assert api.model_info(repo_id).private, "The repository is public. Make it private first."
api.upload_file(path_or_fileobj=GGUF_FILE, path_in_repo=GGUF_FILE, repo_id=repo_id)
print("LOCAL_MODEL_REPO =", repo_id)
print("LOCAL_MODEL_FILE =", GGUF_FILE)

## Next

Put both values in the app's settings, with a **read** token as `HF_TOKEN`. To score real reports, download the model and do it on your computer:

```powershell
python -m training.evaluate <file> --name trained
```